# Dataset Composition (Figure D1)

Replaces the draft's Figures 1 and 2 - nine sub-panels over two full pages - with one
five-panel figure. Panels a-d are the metadata breakdowns; panel e is the question/answer
composition that Figure 2 spent three bar charts on.

Built from `pathopen_data/processed/data.json`, not the raw CSV: the processed records
already normalise magnification (the raw sheet has `5X` and `5x` as separate values) and
express it as the combined microscope x eyepiece figure the paper quotes.


In [ ]:
import os
import sys
import json
import collections

sys.path.insert(0, os.path.abspath("."))

import matplotlib.pyplot as plt
import numpy as np

import nature_style as ns

ns.apply_style()

DATA = os.path.join("..", "pathopen_data", "processed", "data.json")
with open(DATA) as handle:
    cases = json.load(handle)

print(f"cases: {len(cases)}")

BAR = ns.DATASET_COLORS["PathOPEN"]


def top_counts(field, limit=None):
    """Counts for a case-level metadata field, most frequent first.

    Regional Anatomy is free text, so the same entry appears with different casing
    ("Mucosa" / "mucosa"). Fold case before counting, then display the most common
    spelling, or the tail inflates and a real class is split in two."""
    raw = [case[field].strip() for case in cases if case.get(field, "").strip()]
    by_key = collections.defaultdict(list)
    for value in raw:
        by_key[value.lower()].append(value)
    counts = collections.Counter()
    for key, spellings in by_key.items():
        counts[collections.Counter(spellings).most_common(1)[0][0]] = len(spellings)
    ordered = counts.most_common()
    if limit is not None and len(ordered) > limit:
        tail = sum(count for _, count in ordered[limit:])
        ordered = ordered[:limit] + [(f"other ({len(counts) - limit} classes)", tail)]
    return ordered, len(counts)


def hbar(ax, pairs, total_classes, xlabel, wrap=26):
    """Horizontal bars, most frequent at the top, count printed at the end of each."""
    labels = [label[:wrap] for label, _ in pairs][::-1]
    values = [value for _, value in pairs][::-1]
    y = np.arange(len(values))
    colours = [BAR] * len(values)
    if pairs and pairs[-1][0].startswith("other"):
        colours[0] = "#BFBFBF"          # the pooled tail is not a class
    ax.barh(y, values, 0.72, color=colours)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=ns.fs(6.6))
    ax.set_xlabel(xlabel, fontsize=7)
    for index, value in enumerate(values):
        ax.text(value + max(values) * 0.015, index, str(value), va="center",
                fontsize=ns.fs(6.3), color="#444444")
    ax.set_xlim(0, max(values) * 1.14)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    return total_classes


In [ ]:
# --- panel e source: question/answer composition -------------------------------------
# The draft's Figure 2 used three separate bar charts (overall, single-image,
# multi-image) to carry six numbers. One stacked pair carries the same information and
# makes the split that actually matters visible: how many open-ended questions ship with
# a wrong answer as well as a correct one.
single, multi = collections.Counter(), collections.Counter()
for case in cases:
    bucket = single if len(case["Images"]) == 1 else multi
    for item in case["Open Ended"]:
        if not item.get("Question", "").strip():
            continue
        has_wrong = bool(item.get("Wrong Answer", "").strip())
        bucket["correct + wrong" if has_wrong else "correct only"] += 1

n_single = sum(1 for c in cases if len(c["Images"]) == 1)
n_multi = len(cases) - n_single
print(f"single-image cases {n_single}, multi-image {n_multi}")
print("single:", dict(single), " multi:", dict(multi))
print("open-ended questions:", sum(single.values()) + sum(multi.values()))


In [ ]:
fig = plt.figure(figsize=ns.mm(ns.DOUBLE_COL_MM, 158))
outer = fig.add_gridspec(2, 1, height_ratios=[1.0, 1.0], hspace=0.52)

# Row 1: the three case-level fields with few enough classes to show whole.
row1 = outer[0].subgridspec(1, 3, width_ratios=[1.25, 1.0, 0.78], wspace=0.72)

ax = fig.add_subplot(row1[0, 0])
pairs, total = top_counts("Organ", limit=11)
hbar(ax, pairs, total, "Cases")
ax.set_title(f"Organ system ({total} classes)", fontsize=7, pad=3)
ns.panel_label_below(ax, "a")

ax = fig.add_subplot(row1[0, 1])
pairs, total = top_counts("Categorization", limit=9)
hbar(ax, pairs, total, "Cases")
ax.set_title(f"Categorization ({total} classes)", fontsize=7, pad=3)
ns.panel_label_below(ax, "b")

ax = fig.add_subplot(row1[0, 2])
# Magnification is ordinal, so it keeps its natural order rather than being sorted by
# frequency like the nominal fields beside it.
mag_counts = collections.Counter(
    image["Magnification"] for case in cases for image in case["Images"]
    if image.get("Magnification"))
order = sorted(mag_counts, key=lambda m: float(str(m).lower().rstrip("x")))
y = np.arange(len(order))
ax.barh(y, [mag_counts[m] for m in order], 0.72, color=BAR)
ax.set_yticks(y)
ax.set_yticklabels(order, fontsize=ns.fs(7))
ax.invert_yaxis()
ax.set_xlabel("Images", fontsize=7)
ax.set_title("Magnification", fontsize=7, pad=3)
for index, mag in enumerate(order):
    ax.text(mag_counts[mag] + 1.5, index, str(mag_counts[mag]), va="center",
            fontsize=ns.fs(6.3), color="#444444")
ax.set_xlim(0, max(mag_counts.values()) * 1.16)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ns.panel_label_below(ax, "c")

# Row 2: regional anatomy (long tail, so top 15) and the QA composition.
row2 = outer[1].subgridspec(1, 2, width_ratios=[1.15, 1.0], wspace=0.52)

ax = fig.add_subplot(row2[0, 0])
# No pooled "other" bar here: the tail holds 100+ classes and would plot at ~112 against
# real classes of 2-6, flattening everything worth seeing. The remainder is stated in
# the title instead.
SHOW = 14
all_pairs, total = top_counts("Regional Anatomy")
shown = all_pairs[:SHOW]
remainder = sum(count for _, count in all_pairs[SHOW:])
hbar(ax, shown, total, "Cases", wrap=22)
ax.set_title(f"Regional anatomy (top {SHOW} of {total};\n{remainder} cases in the "
             f"remaining {total - SHOW} classes)", fontsize=ns.fs(6.8), pad=3,
             linespacing=1.3)
ns.panel_label_below(ax, "d")

ax = fig.add_subplot(row2[0, 1])
groups = [f"Single-image\n({n_single} cases)", f"Multi-image\n({n_multi} cases)"]
x = np.arange(len(groups))
correct_only = [single["correct only"], multi["correct only"]]
both = [single["correct + wrong"], multi["correct + wrong"]]
ax.bar(x, both, 0.52, color=BAR, label="correct + wrong answer")
ax.bar(x, correct_only, 0.52, bottom=both, color="#BFBFBF",
       label="correct answer only")
for index in range(len(groups)):
    if both[index]:
        ax.text(index, both[index] / 2, str(both[index]), ha="center", va="center",
                fontsize=ns.fs(6.8), color="white")
    if correct_only[index]:
        ax.text(index, both[index] + correct_only[index] / 2, str(correct_only[index]),
                ha="center", va="center", fontsize=ns.fs(6.8), color="#333333")
    total_q = both[index] + correct_only[index]
    ax.text(index, total_q + 6, str(total_q), ha="center", fontsize=ns.fs(6.8), color="#333333")
ax.set_xticks(x)
ax.set_xticklabels(groups, fontsize=ns.fs(7))
ax.set_ylabel("Open-ended questions", fontsize=7)
ax.set_ylim(0, max(b + c for b, c in zip(both, correct_only)) * 1.22)
ns.legend_outside(ax, "above", fontsize=ns.fs(7), ncol=1, bbox_to_anchor=(0.5, 1.02))
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ns.panel_label_below(ax, "e")

paths = ns.save(fig, "figD1_dataset_composition")
print("wrote:", paths)
plt.show()


## Draft caption

**Fig. D1 | Composition of the PathOPEN dataset.**
**a**-**d**, Case-level metadata. Organ system, categorization and regional anatomy are
nominal and ordered by frequency; the grey bar pools the remaining classes. Magnification
is ordinal and keeps its natural order, and is counted per image rather than per case
because a case may carry up to three images at different magnifications. Magnifications
are the combined microscope-objective x eyepiece figure.
**e**, Open-ended question composition. Every question has a pathologist-authored correct
answer; the dark segment counts those that additionally carry a pathologist-authored
*wrong* answer, which is the feature no comparator dataset provides. All multi-image
cases carry both.
